# Phase 2B.4 - Held-Out Final (Kaggle)

Run exactly one approved prompt/context configuration on the frozen held-out set: 284 questions from 50 unseen articles. Held-out results must never be used to revise the winner.

Before execution, attach a `phase2b_winner_decision.json` produced after reviewing both Phase 2B.3 finalist reports. Required contract:

```json
{
  "schema_version": 1,
  "status": "approved",
  "selected_on_partition": "development",
  "heldout_outputs_accessed_for_selection": false,
  "winner": {"prompt_id": "p2", "context_depth": 3},
  "finalist_configs": [
    {"prompt_id": "p2", "context_depth": 5},
    {"prompt_id": "p2", "context_depth": 3}
  ],
  "finalist_result_artifacts": [
    {"name": "finalist_1", "sha256": "..."},
    {"name": "finalist_2", "sha256": "..."}
  ],
  "review": {"reviewer_id": "...", "approved_at": "ISO-8601 timestamp", "notes": "..."}
}
```


In [ ]:
from pathlib import Path
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='c53184ad8046ccf9fca0216d01234125d12bdb12'
HF_ARTIFACT_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-locked-v2'
HF_ARTIFACT_REVISION='locked-bge-m3-512-64-deduplicated-v2'
HF_ARTIFACT_FILENAME='artifacts/locked-bge-m3-512-64-deduplicated-v2/locked-bge-m3-512-64-deduplicated-v2.zip'
HF_ARTIFACT_SHA256='fc5d67b7acf6e8be0205ce00b8069b3b6c8dcce853f8671f2feb3887b2707a24'
HF_PREPARATION_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-experiments'
HF_PREPARATION_REVISION='771fee6e19e3e836c9ba0b15874fe0bd5199b2a5'
HF_PREPARATION_FILENAME='phase2b/preparation-refined-prompts-v1/phase2b_preparation_bundle.zip'
HF_PREPARATION_SHA256='0b2ab8388d6ab679970519bc35ba574936a063b32406ce5cee4b6ec5c166073c'
PREPARATION_REPO_COMMIT='b9d38030e171d4a194d8e92964fae0c8b8aa597a'
PREPARATION_BUNDLE_PATH=''  # Optional local ZIP; otherwise download the private HF bundle.
WINNER_DECISION_PATH=''  # Optional explicit path; otherwise search Kaggle inputs.
RESTORE_CHECKPOINT_PATH=''  # Optional checkpoint created by this notebook.
EXECUTE_API_CALLS=False  # Set True only after attaching the approved decision.
PLATFORM='kaggle'
RUN_ID='phase2b_4_heldout_final'
GENERATOR_SECRET_NAME='GEMINI_API_KEY_1'
GENERATOR_MODEL='gemini-3.1-flash-lite'
GENERATOR_REASONING_EFFORT='minimal'
GENERATOR_MAX_TOKENS=512
GENERATOR_MIN_INTERVAL_SECONDS=4.2
JUDGE_MODEL='accounts/fireworks/models/glm-5p3-flash'
JUDGE_REASONING_EFFORT='low'
JUDGE_MAX_TOKENS=2048
GENERATOR_INPUT_PER_MILLION_USD=0.25
GENERATOR_OUTPUT_PER_MILLION_USD=1.50
JUDGE_INPUT_PER_MILLION_USD=0.15
JUDGE_OUTPUT_PER_MILLION_USD=0.50
TOP_K=20
RERANK_TOP_N=5
SCREENING_QUESTIONS=80  # Frozen upstream split; not executed here.
JUDGE_CALIBRATION_QUESTIONS=20  # Frozen upstream split; not executed here.
FINAL_HELDOUT_ARTICLES=50
REGISTERED_FINALISTS=[{'prompt_id':'p2','context_depth':5},{'prompt_id':'p2','context_depth':3}]
SEED=42


## 1. Environment and immutable inputs

Enable a Kaggle GPU and Internet. Add private secrets `HF_TOKEN`, `GEMINI_API_KEY_1`, and `FIREWORKS_API_KEY`. The HF token is required because the preparation repository is private.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
KAGGLE_INPUT=Path('/kaggle/input'); RUNTIME_ROOT=Path('/kaggle/working')
PROJECT_ROOT=RUNTIME_ROOT/'Text-Mining---NewsQA-RAG'
WORK_ROOT=RUNTIME_ROOT/RUN_ID
DATA_ROOT=WORK_ROOT/'data'; INDEX_ROOT=WORK_ROOT/'index'; BASELINE_ROOT=WORK_ROOT/'baseline'
RUNS_ROOT=WORK_ROOT/'runs'; IDS_ROOT=WORK_ROOT/'question_ids'; PROMPT_ROOT=WORK_ROOT/'prompts'
RESULTS=WORK_ROOT/'results'; LOGS=WORK_ROOT/'logs'; TRACE_ROOT=WORK_ROOT/'heldout_trace'
assert not REPO_COMMIT.startswith('SET_TO_'), 'Pin REPO_COMMIT after committing this notebook'
if not PROJECT_ROOT.exists():
    subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
for package_root in [PROJECT_ROOT/'common',PROJECT_ROOT/'app/backend']:
    if str(package_root) not in sys.path: sys.path.insert(0,str(package_root))
os.environ['PYTHONPATH']=os.pathsep.join([str(PROJECT_ROOT/'common'),str(PROJECT_ROOT/'app/backend'),os.environ.get('PYTHONPATH','')])
os.environ.update({'HF_HOME':str(RUNTIME_ROOT/'hf_cache'),'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','LANGCHAIN_TRACING_V2':'false','LANGSMITH_TRACING':'false','CUDA_VISIBLE_DEVICES':'0'})
from kaggle_secrets import UserSecretsClient
secrets=UserSecretsClient()
def optional_secret(name):
    try: return secrets.get_secret(name) or ''
    except Exception: return ''
HF_TOKEN=optional_secret('HF_TOKEN')
GENERATOR_API_KEY=optional_secret(GENERATOR_SECRET_NAME)
JUDGE_API_KEY=optional_secret('FIREWORKS_API_KEY')
for path in [WORK_ROOT,DATA_ROOT,INDEX_ROOT,BASELINE_ROOT,RUNS_ROOT,IDS_ROOT,PROMPT_ROOT,RESULTS,LOGS,TRACE_ROOT]:
    path.mkdir(parents=True,exist_ok=True)
checkpoint_input=Path(RESTORE_CHECKPOINT_PATH) if RESTORE_CHECKPOINT_PATH else None
if checkpoint_input and checkpoint_input.exists():
    shutil.unpack_archive(checkpoint_input,WORK_ROOT)
    print('Restored checkpoint:',checkpoint_input)
if EXECUTE_API_CALLS:
    assert GENERATOR_API_KEY, f'Configure Kaggle secret {GENERATOR_SECRET_NAME}'
    assert JUDGE_API_KEY, 'Configure Kaggle secret FIREWORKS_API_KEY'
print('Repository commit:',REPO_COMMIT,'| API execution:',EXECUTE_API_CALLS)


In [ ]:
import pandas as pd, yaml
from IPython.display import display
def sha256_file(path,block_size=1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
def write_json(path,value):
    Path(path).write_text(json.dumps(value,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]
def nested(value,path):
    for part in path.split('.'):
        if not isinstance(value,dict) or part not in value: return None
        value=value[part]
    return value
def write_checkpoint():
    checkpoint=RUNTIME_ROOT/f'{RUN_ID}_checkpoint.zip'; temporary=checkpoint.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for name in ['baseline','index','runs','question_ids','prompts','results','logs','heldout_trace']:
            root=WORK_ROOT/name
            if root.exists():
                for path in root.rglob('*'):
                    if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
    temporary.replace(checkpoint)
    print('Checkpoint:',checkpoint,round(checkpoint.stat().st_size/2**20,1),'MiB',flush=True)
    return checkpoint
def run_command(command,label,env_overrides=None):
    command=[str(value) for value in command]
    log_path=LOGS/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('$',' '.join(command),flush=True); print('Log:',log_path,flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        env=os.environ.copy(); env.update(env_overrides or {})
        process=subprocess.Popen(command,cwd=PROJECT_ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line); log.flush()
        code=process.wait()
    if code:
        write_checkpoint()
        raise subprocess.CalledProcessError(code,command)
    return log_path
def successful_ids(path):
    latest={}
    for record in load_jsonl(path): latest[record['question_id']]=record
    return {qid for qid,record in latest.items() if record.get('status')=='success'}


## 2. Validate the frozen preparation and retrieval artifact

The preparation ZIP provides the exact split IDs and prompt texts. The locked dataset provides the 1,152-question deduplicated test set, 22,766 chunks, and BGE-M3 sparse index. Both are verified byte-for-byte before use.


In [ ]:
from huggingface_hub import hf_hub_download
preparation_manifest_path=RESULTS/'preparation_bundle_manifest.json'
if not preparation_manifest_path.exists():
    if PREPARATION_BUNDLE_PATH:
        preparation_zip=Path(PREPARATION_BUNDLE_PATH)
    else:
        assert HF_TOKEN, 'HF_TOKEN is required for the private Phase 2B preparation repository'
        preparation_zip=Path(hf_hub_download(repo_id=HF_PREPARATION_REPO_ID,repo_type='dataset',revision=HF_PREPARATION_REVISION,filename=HF_PREPARATION_FILENAME,token=HF_TOKEN))
    assert preparation_zip.exists() and sha256_file(preparation_zip)==HF_PREPARATION_SHA256, 'Preparation ZIP hash mismatch'
    shutil.unpack_archive(preparation_zip,WORK_ROOT)
preparation_manifest=json.loads(preparation_manifest_path.read_text(encoding='utf-8'))
assert preparation_manifest['schema_version']==1
assert preparation_manifest['repo_commit']==PREPARATION_REPO_COMMIT
for record in preparation_manifest['files']:
    path=WORK_ROOT/record['path']
    assert path.exists() and path.stat().st_size==record['bytes'] and sha256_file(path)==record['sha256'], record['path']
prepared_subset_manifest=json.loads((RESULTS/'subset_manifest.json').read_text(encoding='utf-8'))
assert prepared_subset_manifest['heldout_outputs_accessed_for_selection'] is False
assert prepared_subset_manifest['counts']=={'development':281,'heldout':284,'heldout_reserve':587,'judge_calibration':20,'screening':80,'smoke':5}
assert prepared_subset_manifest['heldout_selection']['method']=='seeded_article_sample'
assert prepared_subset_manifest['heldout_selection']['seed']==46
for name,expected_hash in prepared_subset_manifest['sha256'].items():
    assert sha256_file(IDS_ROOT/f'{name}.json')==expected_hash, f'{name} split hash mismatch'
print('Preparation subset hashes verified')
print('Preparation contract verified:',HF_PREPARATION_REVISION)


In [ ]:
artifact_root=DATA_ROOT/'locked-bge-m3-512-64-deduplicated-v2'
if not (artifact_root/'bundle_manifest.json').exists():
    artifact_zip=Path(hf_hub_download(repo_id=HF_ARTIFACT_REPO_ID,repo_type='dataset',revision=HF_ARTIFACT_REVISION,filename=HF_ARTIFACT_FILENAME,token=HF_TOKEN or None))
    assert sha256_file(artifact_zip)==HF_ARTIFACT_SHA256, 'Locked artifact ZIP hash mismatch'
    artifact_root.mkdir(parents=True,exist_ok=True)
    shutil.unpack_archive(artifact_zip,artifact_root)
bundle_manifest=json.loads((artifact_root/'bundle_manifest.json').read_text(encoding='utf-8'))
assert bundle_manifest['statistics']['chunks']==22766
assert bundle_manifest['statistics']['resolved_questions']==1152
for relative,record in bundle_manifest['artifacts'].items():
    path=artifact_root/relative
    assert path.exists() and path.stat().st_size==record['bytes'] and sha256_file(path)==record['sha256'], relative
testset=artifact_root/'testset_resolved.jsonl'; chunks=artifact_root/'chunks.jsonl'; sparse_index=artifact_root/'bge_m3_sparse.pkl'
test_rows={row['question_id']:row for row in load_jsonl(testset)}
heldout_ids=json.loads((IDS_ROOT/'heldout.json').read_text(encoding='utf-8'))
development_ids=json.loads((IDS_ROOT/'development.json').read_text(encoding='utf-8'))
heldout_reserve_ids=json.loads((IDS_ROOT/'heldout_reserve.json').read_text(encoding='utf-8'))
assert len(test_rows)==1152 and len(heldout_ids)==len(set(heldout_ids))
assert len(heldout_ids)==284
assert len(heldout_reserve_ids)==587
assert set(heldout_ids).isdisjoint(development_ids) and set(heldout_ids).isdisjoint(heldout_reserve_ids)
assert set(development_ids).isdisjoint(heldout_reserve_ids)
assert set(heldout_ids)|set(development_ids)|set(heldout_reserve_ids)==set(test_rows)
heldout_articles={test_rows[qid]['article_key'] for qid in heldout_ids}
assert len(heldout_articles)==50
assert heldout_articles==set(prepared_subset_manifest['heldout_selection']['article_ids'])
print('Locked artifact and held-out partition verified:',len(heldout_articles),'articles /',len(heldout_ids),'questions')


In [ ]:
from newsqa_rag.evaluation.benchmark_io import stable_hash
from newsqa_rag.llm import OpenAILLM
prompt_registry=yaml.safe_load((PROJECT_ROOT/'configs/experiments/phase2_generation_prompts.yaml').read_text(encoding='utf-8'))['prompts']
assert set(prompt_registry)=={'p0','p1','p2','p3'}
assert prompt_registry['p0']['system_prompt']==OpenAILLM.DEFAULT_SYSTEM_PROMPT
for prompt_id,record in prompt_registry.items():
    bundled_prompt=(PROMPT_ROOT/f'{prompt_id}.txt').read_text(encoding='utf-8')
    assert bundled_prompt==record['system_prompt'], f'{prompt_id} differs from the frozen preparation prompt'
config=yaml.safe_load((INDEX_ROOT/'phase2b_config.yaml').read_text(encoding='utf-8'))
config['llm'].update({'model':GENERATOR_MODEL,'temperature':0.0,'max_tokens':GENERATOR_MAX_TOKENS,'reasoning_effort':GENERATOR_REASONING_EFFORT})
config['retrieval'].update({'retriever':'sparse','top_k':TOP_K})
config['retrieval']['sparse'].update({'method':'bge-m3','model':'BAAI/bge-m3','device':'cuda'})
config['retrieval']['reranker'].update({'enabled':True,'type':'cross-encoder','model':'BAAI/bge-reranker-large','top_n':RERANK_TOP_N,'batch_size':8,'device':'cuda'})
config_path=INDEX_ROOT/'heldout_config.yaml'
config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
profile=json.loads((INDEX_ROOT/'phase2b_variant.json').read_text(encoding='utf-8'))
config_hash=stable_hash(config)
profile['pipeline'].update({'config_path':str(config_path),'config_sha256':config_hash})
profile['database'].update({'indexed':False,'chunk_count':22766})
profile['artifacts']['chunks']={'path':str(chunks),'bytes':chunks.stat().st_size,'sha256':sha256_file(chunks)}
profile['artifacts']['testset_resolved']={'path':str(testset),'bytes':testset.stat().st_size,'sha256':sha256_file(testset)}
profile['artifacts']['bm25']={'path':str(sparse_index),'bytes':sparse_index.stat().st_size,'sha256':sha256_file(sparse_index)}
profile_path=INDEX_ROOT/'heldout_variant.json'; write_json(profile_path,profile)
assert config['chunking']['chunk_size']==512 and config['chunking']['chunk_overlap']==64
assert config['retrieval']['sparse']['method']=='bge-m3'
assert config['retrieval']['reranker']['model']=='BAAI/bge-reranker-large'
display(pd.DataFrame([{'prompt_id':key,'name':value['name'],'sha256':sha256_file(PROMPT_ROOT/f'{key}.txt')} for key,value in prompt_registry.items()]))


## 3. Validate the winner decision

This cell accepts the reviewed decision but does not inspect finalist scores or select a winner. If no decision is attached, it emits a template and leaves execution locked.


In [ ]:
decision_path=Path(WINNER_DECISION_PATH) if WINNER_DECISION_PATH else None
if decision_path is None and (RESULTS/'winner_decision.json').exists():
    decision_path=RESULTS/'winner_decision.json'
if decision_path is None:
    candidates=sorted(KAGGLE_INPUT.rglob('phase2b_winner_decision.json'))
    decision_path=candidates[-1] if candidates else None
decision_template={'schema_version':1,'status':'pending','selected_on_partition':'development','heldout_outputs_accessed_for_selection':False,'winner':{'prompt_id':None,'context_depth':None},'finalist_configs':REGISTERED_FINALISTS,'finalist_result_artifacts':[{'name':'finalist_1','sha256':''},{'name':'finalist_2','sha256':''}],'review':{'reviewer_id':'','approved_at':'','notes':''}}
if decision_path is None:
    template_path=RUNTIME_ROOT/'phase2b_winner_decision_TEMPLATE.json'
    write_json(template_path,decision_template)
    winner_decision=None; locked_winner=None; decision_sha256=None
    print('Winner decision is not attached. Template:',template_path)
else:
    winner_decision=json.loads(decision_path.read_text(encoding='utf-8'))
    assert winner_decision.get('schema_version')==1
    assert winner_decision.get('status')=='approved'
    assert winner_decision.get('selected_on_partition')=='development'
    assert winner_decision.get('heldout_outputs_accessed_for_selection') is False
    locked_winner=winner_decision.get('winner',{})
    assert set(locked_winner)=={'prompt_id','context_depth'}
    assert locked_winner['prompt_id'] in prompt_registry
    assert locked_winner['context_depth'] in {1,3,5}
    assert winner_decision.get('finalist_configs')==REGISTERED_FINALISTS
    assert locked_winner in REGISTERED_FINALISTS
    finalist_artifacts=winner_decision.get('finalist_result_artifacts',[])
    assert len(finalist_artifacts)==2
    assert len({record.get('name') for record in finalist_artifacts})==len(finalist_artifacts)
    assert all(record.get('name') and len(record.get('sha256',''))==64 and all(char in '0123456789abcdef' for char in record['sha256'].lower()) for record in finalist_artifacts)
    review=winner_decision.get('review',{})
    assert review.get('reviewer_id') and review.get('approved_at')
    decision_sha256=sha256_file(decision_path)
    if decision_path.resolve()!=(RESULTS/'winner_decision.json').resolve(): shutil.copy2(decision_path,RESULTS/'winner_decision.json')
    print('Approved winner:',locked_winner,'| decision SHA-256:',decision_sha256)


## 4. One-time held-out execution

The run is resumable. The access record is written before retrieval begins; reruns must use the same decision hash and winner. Retrieval is BGE-M3 sparse top-20 plus BGE-large reranking top-5. Generation uses the frozen reranked trace and only the approved context depth.


In [ ]:
assert EXECUTE_API_CALLS, 'Set EXECUTE_API_CALLS=True only after attaching an approved winner decision'
assert winner_decision is not None and locked_winner is not None
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU for BGE-M3 query encoding and BGE-large reranking'
access_path=RESULTS/'heldout_access.json'
access_contract={'schema_version':1,'decision_sha256':decision_sha256,'winner':locked_winner,'heldout_ids_sha256':sha256_file(IDS_ROOT/'heldout.json'),'article_count':50,'question_count':284,'artifact_sha256':HF_ARTIFACT_SHA256,'preparation_sha256':HF_PREPARATION_SHA256}
if access_path.exists():
    previous_access=json.loads(access_path.read_text(encoding='utf-8'))
    for key,value in access_contract.items(): assert previous_access.get(key)==value, f'Held-out lock mismatch: {key}'
else:
    write_json(access_path,{**access_contract,'status':'started','started_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())})
heldout_retrievals=TRACE_ROOT/'retrievals.jsonl'
retrieval_command=[sys.executable,'-u','scripts/collect_benchmark_predictions.py','--retriever','sparse','--reranker','cross-encoder','--reranker-model','BAAI/bge-reranker-large','--testset',testset,'--variant-manifest',profile_path,'--config',config_path,'--run-dir',TRACE_ROOT,'--question-ids-file',IDS_ROOT/'heldout.json','--top-k',TOP_K,'--rerank-top-n',RERANK_TOP_N,'--seed',SEED,'--retrieval-only','--max-attempts',3,'--retry-failed','--progress']
run_command(retrieval_command,'heldout_retrieval')
assert successful_ids(heldout_retrievals)==set(heldout_ids), 'Held-out retrieval trace is incomplete'
prompt_id=locked_winner['prompt_id']; context_depth=locked_winner['context_depth']
run_dir=RUNS_ROOT/f'heldout_final__{prompt_id}__d{context_depth}'
generation_command=[sys.executable,'-u','scripts/collect_benchmark_predictions.py','--retriever','sparse','--reranker','cross-encoder','--reranker-model','BAAI/bge-reranker-large','--testset',testset,'--variant-manifest',profile_path,'--config',config_path,'--run-dir',run_dir,'--question-ids-file',IDS_ROOT/'heldout.json','--top-k',TOP_K,'--rerank-top-n',RERANK_TOP_N,'--generator-model',GENERATOR_MODEL,'--prompt-id',prompt_id,'--system-prompt-file',PROMPT_ROOT/f'{prompt_id}.txt','--context-depth',context_depth,'--source-retrievals',heldout_retrievals,'--generation-min-interval-seconds',GENERATOR_MIN_INTERVAL_SECONDS,'--seed',SEED,'--max-attempts',3,'--retry-failed','--progress']
run_command(generation_command,'heldout_generation',{'GEMINI_API_KEY':GENERATOR_API_KEY})
assert successful_ids(run_dir/'predictions.jsonl')==set(heldout_ids), 'Held-out generation is incomplete'
run_command([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',run_dir],'heldout_score_prejudge')
judge_command=[sys.executable,'-u','scripts/judge_benchmark_predictions.py','--run-dir',run_dir,'--judge-provider','fireworks','--judge-model',JUDGE_MODEL,'--reasoning-effort',JUDGE_REASONING_EFFORT,'--judge-max-tokens',JUDGE_MAX_TOKENS,'--question-ids-file',IDS_ROOT/'heldout.json','--batch-size',1,'--max-workers',1,'--seed',SEED,'--max-attempts',3,'--retry-failed','--require-complete-metrics','--progress']
run_command(judge_command,'heldout_judge',{'FIREWORKS_API_KEY':JUDGE_API_KEY})
run_command([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',run_dir],'heldout_score_final')
assert successful_ids(run_dir/'judge_results.jsonl')==set(heldout_ids), 'Held-out judging is incomplete'
started_at=json.loads(access_path.read_text(encoding='utf-8')).get('started_at')
write_json(access_path,{**access_contract,'status':'complete','started_at':started_at,'completed_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'retrievals_sha256':sha256_file(heldout_retrievals),'predictions_sha256':sha256_file(run_dir/'predictions.jsonl'),'judge_results_sha256':sha256_file(run_dir/'judge_results.jsonl')})
print('Held-out final completed:',run_dir)


## 5. Final report and export

This section reports the single held-out estimate. It does not compare configurations or feed a result back into winner selection. Download both the results ZIP and checkpoint.


In [ ]:
assert (RESULTS/'heldout_access.json').exists() and json.loads((RESULTS/'heldout_access.json').read_text(encoding='utf-8'))['status']=='complete'
report=json.loads((run_dir/'report.json').read_text(encoding='utf-8'))
assert nested(report,'coverage.expected')==284 and nested(report,'coverage.successful')==284
assert nested(report,'ragas.n_samples')==284
judge_batches={}
for record in load_jsonl(run_dir/'judge_results.jsonl'):
    judge_batches[record.get('batch_id',record['question_id'])]=record.get('batch_usage',{})
judge_input=sum(value.get('input_tokens',0) for value in judge_batches.values())
judge_output=sum(value.get('output_tokens',0) for value in judge_batches.values())
judge_requests=sum(value.get('successful_requests',0) for value in judge_batches.values())
generation_input=nested(report,'usage.input_tokens') or 0; generation_output=nested(report,'usage.output_tokens') or 0
generation_cost=generation_input/1e6*GENERATOR_INPUT_PER_MILLION_USD+generation_output/1e6*GENERATOR_OUTPUT_PER_MILLION_USD
judge_cost=judge_input/1e6*JUDGE_INPUT_PER_MILLION_USD+judge_output/1e6*JUDGE_OUTPUT_PER_MILLION_USD
question_rows=[]
for row in load_jsonl(run_dir/'deterministic_scores.jsonl'):
    question_rows.append({'question_id':row['question_id'],'article_key':row['article_key'],'retrieval_group':'gold_in_top5' if nested(row,'retrieval.hit_rate@5')==1 else 'gold_not_in_top5','qa_f1':nested(row,'qa.f1'),'citation_f1':nested(row,'citations.citation_f1'),'citation_validity':nested(row,'citations.citation_validity'),'answer_correctness':nested(row,'ragas.answer_correctness'),'faithfulness':nested(row,'ragas.faithfulness'),'answer_relevancy':nested(row,'ragas.answer_relevancy')})
question_frame=pd.DataFrame(question_rows)
metric_columns=['qa_f1','citation_f1','citation_validity','answer_correctness','faithfulness','answer_relevancy']
assert len(question_frame)==284 and question_frame[metric_columns].notna().all().all()
article_frame=question_frame.groupby('article_key',as_index=False)[metric_columns].mean()
assert len(article_frame)==50
subgroup_frame=question_frame.groupby('retrieval_group')[metric_columns].agg(['count','mean']).reset_index()
question_frame.to_csv(RESULTS/'heldout_question_scores.csv',index=False)
article_frame.to_csv(RESULTS/'heldout_article_scores.csv',index=False)
subgroup_frame.to_csv(RESULTS/'heldout_retrieval_subgroups.csv',index=False)
article_macro={column:round(float(article_frame[column].mean()),4) for column in metric_columns}
summary={'schema_version':1,'partition':'heldout','articles':50,'questions':284,'winner':locked_winner,'decision_sha256':decision_sha256,'retrieval':{'method':'bge-m3-sparse','top_k':TOP_K,'reranker':'BAAI/bge-reranker-large','rerank_top_n':RERANK_TOP_N},'generator':{'model':GENERATOR_MODEL,'reasoning_effort':GENERATOR_REASONING_EFFORT,'max_tokens':GENERATOR_MAX_TOKENS},'judge':{'provider':'fireworks','model':JUDGE_MODEL,'reasoning_effort':JUDGE_REASONING_EFFORT,'max_tokens':JUDGE_MAX_TOKENS},'metrics_question_micro':{'qa_exact_match':nested(report,'qa.exact_match'),'qa_f1':nested(report,'qa.f1'),'citation_f1':nested(report,'citations.citation_f1'),'citation_validity':nested(report,'citations.citation_validity'),'answer_correctness':nested(report,'ragas.answer_correctness'),'faithfulness':nested(report,'ragas.faithfulness'),'answer_relevancy':nested(report,'ragas.answer_relevancy')},'metrics_article_macro':article_macro,'coverage':report['coverage'],'usage':{'generation_input_tokens':generation_input,'generation_output_tokens':generation_output,'judge_input_tokens':judge_input,'judge_output_tokens':judge_output,'judge_successful_requests':judge_requests},'cost_usd':{'generation':round(generation_cost,6),'judge':round(judge_cost,6),'total':round(generation_cost+judge_cost,6),'pricing_assumption_per_million_tokens':{'generator_input':GENERATOR_INPUT_PER_MILLION_USD,'generator_output':GENERATOR_OUTPUT_PER_MILLION_USD,'judge_input':JUDGE_INPUT_PER_MILLION_USD,'judge_output':JUDGE_OUTPUT_PER_MILLION_USD}},'latency':{'generation_p50_ms':nested(report,'latency.llm_ms.p50_ms'),'generation_p95_ms':nested(report,'latency.llm_ms.p95_ms'),'total_p50_ms':nested(report,'latency.total.p50_ms'),'total_p95_ms':nested(report,'latency.total.p95_ms')},'provenance':{'repo_commit':REPO_COMMIT,'locked_artifact_repo':HF_ARTIFACT_REPO_ID,'locked_artifact_revision':HF_ARTIFACT_REVISION,'locked_artifact_sha256':HF_ARTIFACT_SHA256,'preparation_repo':HF_PREPARATION_REPO_ID,'preparation_revision':HF_PREPARATION_REVISION,'preparation_sha256':HF_PREPARATION_SHA256,'heldout_ids_sha256':sha256_file(IDS_ROOT/'heldout.json'),'prompt_sha256':sha256_file(PROMPT_ROOT/f'{prompt_id}.txt'),'collector_run_fingerprint':json.loads((run_dir/'run_manifest.json').read_text(encoding='utf-8'))['run_fingerprint']},'completed_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
write_json(RESULTS/'heldout_final_summary.json',summary)
pd.json_normalize(summary,sep='.').to_csv(RESULTS/'heldout_final_summary.csv',index=False)
display(pd.DataFrame([summary['metrics_question_micro']]))
display(subgroup_frame)
checkpoint=write_checkpoint()
result_bundle=RUNTIME_ROOT/f'{RUN_ID}_results.zip'; temporary=result_bundle.with_suffix('.zip.tmp')
with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ['runs','question_ids','prompts','results','logs','heldout_trace']:
        root=WORK_ROOT/name
        if root.exists():
            for path in root.rglob('*'):
                if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
temporary.replace(result_bundle)
print('Results:',result_bundle,round(result_bundle.stat().st_size/2**20,1),'MiB')
print('Checkpoint:',checkpoint,round(checkpoint.stat().st_size/2**20,1),'MiB')
